# **Cross-Fitting T-learner**

The T-learner (Two learner) solves the S-learner's biggest weakness; where tree-based models can accidentally ignore the treatment variable ($X$) during training because powerful confounders ($Z$) overshadow it. By training two completely independent models, the T-learner forces the algorithm to account for the treatment.

## **The Churn Scenario Setup**

Imagine you run a subscription software company and want to know: Does offering a $20 discount code ($X$) actually prevent customers from churning ($Y$)?

In your historical data, you have:
- Treatment ($X$): $1$ if the customer received a discount, $0$ if they did not.
- Outcome ($Y$): $1$ if the customer churned (canceled), $0$ if they stayed.- Confounders ($Z$): Variables that affect both whether a customer gets a discount and whether they churn.
    - Tenure (months subscribed)
    - Weekly app logins
    - Support tickets
    - Monthly spend tier

## **Implementation**

### **Training and Testing of the Model**

In [16]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split,GridSearchCV, StratifiedKFold
from sklearn.metrics import  recall_score, precision_score, f1_score,roc_auc_score

**Generate synthetic data for demonstration**

In [17]:
def generate_enterprise_churn_data(n=10000, seed=42):
    np.random.seed(seed)
    
    # 1. Realistic Confounders
    # Tenure: Gamma distribution (skewed: most customers are newer, fewer are multi-year veterans)
    tenure_months = (np.clip(np.random.gamma(shape=2, scale=6, size=n), 1, 60)).astype(int)
    
    # Weekly Logins: Poisson distribution
    weekly_logins = np.random.poisson(lam=4, size=n)
    
    # Support Tickets: Higher tickets usually indicate friction
    support_tickets = np.random.poisson(lam=1.5, size=n)
    
    # Monthly Spend Tier ($10, $25, $50, $100)
    monthly_spend = np.random.choice([10, 25, 50, 100], size=n, p=[0.4, 0.3, 0.2, 0.1])
    
    # 2. Confounding Mechanism (Propensity Score)
    # Retention teams historically give discounts to customers with low tenure, low logins, AND high support tickets.
    propensity_logit = -0.5 + (0.05 * tenure_months) - (0.20 * weekly_logins) + (0.40 * support_tickets) - (0.005 * monthly_spend)
    discount_prob = 1 / (1 + np.exp(-propensity_logit))
    treatment_x = np.random.binomial(1, discount_prob)
    
    # 3. Outcome Mechanism (Churn)
    # True Causal Effect: Offering a discount reduces the log-odds of churning by -0.35 (~15-20% absolute drop)
    true_causal_effect = -0.35
    
    baseline_logit = (
        1.2 
        - (0.06 * tenure_months) 
        - (0.15 * weekly_logins) 
        + (0.50 * support_tickets) 
        - (0.01 * monthly_spend)
        + (true_causal_effect * treatment_x) # The true causal impact!
    )
    churn_prob = 1 / (1 + np.exp(-baseline_logit))
    outcome_y = np.random.binomial(1, churn_prob)
    
    df = pd.DataFrame({
        'tenure_months': tenure_months,
        'weekly_logins': weekly_logins,
        'support_tickets': support_tickets,
        'monthly_spend': monthly_spend,
        'discount_given': treatment_x, # Treatment (X)
        'churned': outcome_y           # Outcome (Y)
    })
    
    return df

In [18]:
df = generate_enterprise_churn_data(n=10000)

In [19]:
df.head()

,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,churned
0,14,7,1,25,0,0
1,8,4,3,50,0,0
2,8,9,2,10,1,1
3,8,7,0,10,0,0
4,27,3,1,100,0,1


**Split the data**

In [20]:
# train test split
X = df.drop(columns=['churned'])
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

In [21]:
print(f"Dataset Split Sizes:")
print(f" - Train Set: {len(X_train)} rows")
print(f" - Test Set: {len(X_test)} rows\n")

Dataset Split Sizes:
 - Train Set: 8000 rows
 - Test Set: 2000 rows



**Prepare Subsets for T-Learner**

In [22]:
# Note: In a T-learner, 'discount_given' is EXCLUDED from the feature set 
# because the treatment is split into two separate models entirely.
features = ['tenure_months', 'weekly_logins', 'support_tickets', 'monthly_spend']

**Cross-fitting T-Learner**

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
all_individual_effects = []
print("Starting T-Learner Cross-Fitting across 5 folds...")

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    # Split fold data
    X_tr = X_train.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    
    # Inside the training fold, split into Control (X=0) and Treatment (X=1) subsets
    train_mask_0 = (X_tr['discount_given'] == 0)
    train_mask_1 = (X_tr['discount_given'] == 1)

    X_train_0 = X_tr.loc[train_mask_0, features]
    y_train_0 = y_tr[train_mask_0]

    X_train_1 = X_tr.loc[train_mask_1, features]
    y_train_1 = y_tr[train_mask_1]

    # Train separate models for Control and Treatment folds
    # You can use different values for n_estimators, you would see that the ATE varies for each value
    fold_f0 = RandomForestClassifier(max_depth=10, n_estimators=250, random_state=50)
    fold_f0.fit(X_train_0, y_train_0)
    
    fold_f1 = RandomForestClassifier(max_depth=10, n_estimators=250, random_state=50)
    fold_f1.fit(X_train_1, y_train_1)

    # Counterfactual predictions on the validation fold
    X_val_features = X_val[features]
    prob_y1 = fold_f1.predict_proba(X_val_features)[:, 1]  # P(Y=1 | X=1) Treatment
    prob_y0 = fold_f0.predict_proba(X_val_features)[:, 1]  # P(Y=1 | X=0) Control
    
    # Calculate fold causal effects and store them
    fold_effects = prob_y1 - prob_y0
    all_individual_effects.extend(fold_effects)



print("Cross-Fitting Complete!\n")

# Calculate Cross-Fitted Training ATE
cross_fitted_ate = np.round(np.mean(all_individual_effects), 3)
print(f"Cross-Fitted Training ATE (T-Learner - Validation Set): {cross_fitted_ate}\n")


Starting T-Learner Cross-Fitting across 5 folds...
Cross-Fitting Complete!

Cross-Fitted Training ATE (T-Learner - Validation Set): -0.065



**Final Training of T-Models**

In [24]:
train_mask_0 = (X_train['discount_given'] == 0) # Control 
train_mask_1 = (X_train['discount_given'] == 1) # Treatment

X_train_0 = X_train.loc[train_mask_0, features]
y_train_0 = y_train[train_mask_0]

X_train_1 = X_train.loc[train_mask_1, features]
y_train_1 = y_train[train_mask_1]

# Train separate models for Control and Treatment folds
final_f0 = RandomForestClassifier(max_depth=10, n_estimators=150, random_state=50)
final_f0.fit(X_train_0, y_train_0)

final_f1 = RandomForestClassifier(max_depth=10, n_estimators=250, random_state=50)
final_f1.fit(X_train_1, y_train_1)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",250
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

### **Counterfactuals (ATE): T-Learner Approach**

**Simulate A/B Counterfactuals on Test Set**

In [25]:
X_test_features = X_test[features]

# World A: Predict churn probability if EVERYONE got the discount (using Model f1)
prob_y1 = final_f1.predict_proba(X_test_features)[:, 1]

# World B: Predict churn probability if NO ONE got the discount (using Model f0)
prob_y0 = final_f0.predict_proba(X_test_features)[:, 1]

# Calculate Average Treatment Effect (ATE)
# This is like from s-learner: prob_y1 - prob_y2;
# where prob_y1 is the predicted churn probability if treated with discount, 
# and prob_y0 is the predicted churn probability if not treated.
ate = np.round(np.mean(prob_y1 - prob_y0), 3)
print(f"Average Treatment Effect (ATE) of Discount on Churn Probability (T-Learner):\n{ate}")

Average Treatment Effect (ATE) of Discount on Churn Probability (T-Learner):
-0.069


### **Inspection of Prediction Groups**

**Inspection of Prediction group A**

In [26]:
# Inspection of Counterfactual Group A (Trained with Treatment = 1) Gave discount
test_results_f1 = X_test.copy()
test_results_f1['predicted_churn_prob_if_treated'] = final_f1.predict(X_test_features)
test_results_f1['actual_churn'] = y_test
print("--- Counterfactual Group A Preview ---")
test_results_f1.head()

--- Counterfactual Group A Preview ---


,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,predicted_churn_prob_if_treated,actual_churn
9102,35,6,1,25,0,0,0
7868,5,2,2,50,0,1,1
4176,5,2,0,100,0,0,0
4161,4,2,1,10,1,1,1
8770,6,4,1,25,0,0,1


**Inspection of Prediction group B**

In [27]:
# Inspection of Counterfactual Group B (Trained with Treatment = 0) # No discount
test_results_f0 = X_test.copy()
test_results_f0['predicted_churn_prob_if_control'] = final_f0.predict(X_test_features)
test_results_f0['actual_churn'] = y_test
print("--- Counterfactual Group B Preview ---")
test_results_f0.head()

--- Counterfactual Group B Preview ---


,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,predicted_churn_prob_if_control,actual_churn
9102,35,6,1,25,0,0,0
7868,5,2,2,50,0,1,1
4176,5,2,0,100,0,0,0
4161,4,2,1,10,1,1,1
8770,6,4,1,25,0,1,1


### **Interpretation of ATE**

Because the T-learner isolates the treatment groups into two entirely separate models ($f_0$ and $f_1$), this **-6.9%** absolute drop is completely immune to the S-learner's risk of a tree algorithm accidentally ignoring the discount column.

Since estimated Average Treatment Effect (ATE) is **-0.069** (or roughly **-6.9%**), it means:

**What This Means for Your Business:**

1. **Stronger Causal Proof:** We can confidently tell stakeholders: *"When we give eligible high-risk customers a $20 discount, it causes an absolute **6.9 percentage point decrease** in their probability of churning."*
2. **Real-World Translation:** If a specific customer segment has a baseline predicted churn rate of **30%**, giving them the discount reduces their expected churn risk to **23.1%** ($30\% - 6.9\%$).
3. **Model Validation:** The fact that both the S-learner (-5.8%) and the T-learner (-6.9%) land in the same neighborhood tells us the causal signal is robust, stable, and not dependent on how the model was structured.